# LeetCode 3320 — Count the Number of Winning Sequences bằng Python

- Link: https://leetcode.com/problems/count-the-number-of-winning-sequences/
- Độ khó: **Hard**
- Chủ đề: **DP theo (vòng, nước đi cuối của Bob, hiệu điểm)**, dịch mảng theo hiệu `±1`, cắt tỉa trạng thái "chắc thắng / chắc thua"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countWinningSequences(self, s: str) -> int:
```

## 1. Mục tiêu học

1. Mã hóa luật "oẳn tù tì" 3 sinh vật: `F` thắng `E`, `W` thắng `F`, `E` thắng `W`.
2. Chỉ cần **hiệu điểm** `Bob − Alice` (không cần từng điểm riêng) ⇒ trạng thái `(i, nước cuối của Bob, hiệu)`.
3. Ràng buộc "Bob không lặp nước đi liền nhau" ⇒ chuyển từ 2 nước cuối khác.
4. Cài DP bằng **mảng dịch** (offset `n`) để mỗi bước là vài phép cộng vector.
5. Cắt tỉa: hiệu `> n − i` chắc thắng, hiệu `< −(n − i)` chắc thua.

## 2. Đề bài & ví dụ

Alice và Bob chơi `n` vòng; mỗi vòng mỗi người triệu hồi `F` (rồng lửa), `W` (rắn nước) hoặc `E` (người đá). Luật: `F` gặp `E` thì `F` được 1 điểm; `W` gặp `F` thì `W` được 1; `E` gặp `W` thì `E` được 1; giống nhau thì không ai được điểm. Biết chuỗi nước đi `s` của Alice; Bob **không bao giờ** dùng cùng một sinh vật ở hai vòng liên tiếp. Đếm số chuỗi nước đi của Bob mà **tổng điểm Bob > tổng điểm Alice**, modulo `10^9 + 7`.

**Ví dụ 1:** `"FFF"` → `3` (`"WFW"`, `"FWF"`, `"WEW"`). **Ví dụ 2:** `"FWEFW"` → `18`.

**Ràng buộc:** `1 ≤ n ≤ 1000`.

In [1]:
from typing import List
from itertools import product
MOD = 10 ** 9 + 7

examples = [("FFF", 3), ("FWEFW", 18)]
for s, exp in examples:
    print(repr(s), '-> kỳ vọng', exp)

'FFF' -> kỳ vọng 3
'FWEFW' -> kỳ vọng 18


## 3. Kiến thức nền

### 3.1. Mã hóa luật thắng

Đặt `F = 0`, `W = 1`, `E = 2`. `W` thắng `F`, `E` thắng `W`, `F` thắng `E` ⇒ `b` thắng `a` ⇔ `b ≡ a + 1 (mod 3)`. Điểm hiệu của một vòng: `+1` nếu Bob thắng, `−1` nếu Alice thắng, `0` nếu hòa.

In [2]:
IDX = {'F': 0, 'W': 1, 'E': 2}

def gain(b, a):
    """Hiệu điểm (Bob − Alice) của một vòng."""
    if b == a:
        return 0
    return 1 if b == (a + 1) % 3 else -1

print('bảng gain[b][a] (hàng = Bob F,W,E; cột = Alice F,W,E):')
for b in range(3):
    print('  ', 'FWE'[b], [gain(b, a) for a in range(3)])

bảng gain[b][a] (hàng = Bob F,W,E; cột = Alice F,W,E):
   F [0, -1, 1]
   W [1, 0, -1]
   E [-1, 1, 0]


### 3.2. Hiệu điểm và offset

Hiệu nằm trong `[−n, n]` ⇒ lưu ở chỉ số `hiệu + n` của mảng dài `2n + 1`. Cộng `+1` vào hiệu = dịch mảng sang phải một ô.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

`"FFF"`: Bob thắng `F` bằng `W` (+1), hòa bằng `F` (0), thua bằng `E` (−1). Bob cần tổng `> 0` và không lặp liền:

- `W F W`: `+1, 0, +1` = 2 ✓; `F W F`: `0, +1, 0` = 1 ✓; `W E W`: `+1, −1, +1` = 1 ✓.
- `W W ?` bị cấm (lặp). `W F E`: `+1, 0, −1 = 0` — hòa, không thắng.

⇒ 3 ✓. Thông tin cần nhớ khi đi tiếp: nước cuối của Bob (cho luật cấm lặp) và hiệu hiện tại.

## 5. Cách 1 — Brute Force: mọi chuỗi của Bob

In [3]:
def brute(s):
    total = 0
    for seq in product(range(3), repeat=len(s)):
        if any(seq[i] == seq[i + 1] for i in range(len(s) - 1)):
            continue
        if sum(gain(b, IDX[a]) for b, a in zip(seq, s)) > 0:
            total += 1
    return total % MOD

for s, exp in examples:
    print(brute(s), '| kỳ vọng', exp)

3 | kỳ vọng 3
18 | kỳ vọng 18


**Độ phức tạp:** `O(3^n · n)`.

## 6. Ý tưởng thuật toán / bước đột phá

`dp[b][d]` = số chuỗi của Bob cho các vòng đã xét, kết thúc bằng nước `b`, có hiệu `d`. Chuyển sang vòng có Alice đi `a`:

```
new[b][d + gain(b, a)] += dp[p][d]    với mọi p ≠ b
```

Đáp án `= Σ_b Σ_{d > 0} dp[b][d]`. Trạng thái `3 · (2n + 1)`, mỗi vòng `O(n)` ⇒ tổng `O(n²)` ≈ `6·10^6` phép cộng — làm bằng phép cộng vector đã dịch.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ `(i, last, diff)`

In [4]:
from functools import lru_cache
import sys
sys.setrecursionlimit(10000)

def memo(s):
    n = len(s)
    A = [IDX[c] for c in s]

    @lru_cache(maxsize=None)
    def f(i, last, diff):
        if diff + (n - i) <= 0:              # kể cả thắng hết các vòng còn lại cũng không > 0
            return 0
        if i == n:
            return 1
        return sum(f(i + 1, b, diff + gain(b, A[i])) for b in range(3) if b != last) % MOD

    ans = f(0, -1, 0)
    f.cache_clear()
    return ans

for s, exp in examples:
    print(memo(s), '| kỳ vọng', exp)

3 | kỳ vọng 3
18 | kỳ vọng 18


Cắt tỉa `diff + (n − i) ≤ 0` rất hữu ích: không thể vượt lên nữa ⇒ trả 0 ngay.

### 7.2. Cách 3 — DP lặp với mảng dịch (lời giải chính)

In [5]:
def dp_shift(s):
    n = len(s); off = n; W = 2 * n + 1
    dp = [[0] * W for _ in range(3)]
    a = IDX[s[0]]
    for b in range(3):
        dp[b][off + gain(b, a)] += 1
    for ch in s[1:]:
        a = IDX[ch]
        nd = [[0] * W for _ in range(3)]
        for b in range(3):
            g = gain(b, a)
            p1, p2 = dp[(b + 1) % 3], dp[(b + 2) % 3]      # hai nước khác b
            if g == 1:
                nd[b][1:] = [(u + v) % MOD for u, v in zip(p1[:-1], p2[:-1])]
            elif g == 0:
                nd[b] = [(u + v) % MOD for u, v in zip(p1, p2)]
            else:
                nd[b][:-1] = [(u + v) % MOD for u, v in zip(p1[1:], p2[1:])]
        dp = nd
    return sum(sum(dp[b][off + 1:]) for b in range(3)) % MOD

for s, exp in examples:
    print(dp_shift(s), '| kỳ vọng', exp)

3 | kỳ vọng 3
18 | kỳ vọng 18


### 7.3. Cách 4 — DP với cắt tỉa "chắc thắng" / "chắc thua"

Sau vòng `i` (còn `r = n − i` vòng): nếu hiệu `d > r` thì kể cả thua hết vẫn thắng ⇒ gộp vào bộ đếm `win[b]` (số chuỗi đã chắc thắng, kết thúc bằng `b`); số cách đi tiếp chỉ còn phụ thuộc luật không lặp: mỗi vòng còn lại có **2** lựa chọn ⇒ nhân `2^r`. Nếu `d ≤ −r` thì chắc không thắng ⇒ bỏ. Chỉ giữ hiệu trong `(−r, r]`.

In [6]:
def dp_pruned(s):
    n = len(s)
    A = [IDX[c] for c in s]
    cur = [dict() for _ in range(3)]          # cur[b][d] = số cách
    sure = 0                                    # số chuỗi (tiền tố) chắc thắng, chưa nhân phần đuôi
    for b in range(3):
        cur[b][gain(b, A[0])] = 1
    def prune(i):                               # sau vòng i (đã đi i+1 vòng), còn r vòng
        nonlocal sure
        r = n - 1 - i
        for b in range(3):
            for d in list(cur[b]):
                if d > r:
                    sure = (sure + cur[b][d] * pow(2, r, MOD)) % MOD
                    del cur[b][d]
                elif d <= -r:
                    del cur[b][d]
    prune(0)
    for i in range(1, n):
        a = A[i]
        nxt = [dict() for _ in range(3)]
        for b in range(3):
            g = gain(b, a)
            for p in range(3):
                if p != b:
                    for d, v in cur[p].items():
                        nxt[b][d + g] = (nxt[b].get(d + g, 0) + v) % MOD
        cur = nxt
        prune(i)
    return (sure + sum(v for b in range(3) for d, v in cur[b].items() if d > 0)) % MOD

for s, exp in examples:
    print(dp_pruned(s), '| kỳ vọng', exp)

3 | kỳ vọng 3
18 | kỳ vọng 18


Sau vòng cuối `r = 0`: trạng thái `d > 0` bị gộp vào `sure` (nhân `2^0 = 1`), `d ≤ 0` bị bỏ.

## 8. TRACE từng bước

Trace cách 3 trên `"FFF"` (in các hiệu khác 0 cho từng nước cuối).

In [7]:
def trace(s):
    n = len(s); off = n; W = 2 * n + 1
    show = lambda row: {d - off: v for d, v in enumerate(row) if v}
    dp = [[0] * W for _ in range(3)]
    for b in range(3):
        dp[b][off + gain(b, IDX[s[0]])] += 1
    print('vòng 0:', {('FWE'[b]): show(dp[b]) for b in range(3)})
    for i, ch in enumerate(s[1:], 1):
        a = IDX[ch]
        nd = [[0] * W for _ in range(3)]
        for b in range(3):
            g = gain(b, a)
            for p in range(3):
                if p != b:
                    for d in range(W):
                        if dp[p][d] and 0 <= d + g < W:
                            nd[b][d + g] += dp[p][d]
        dp = nd
        print(f'vòng {i}:', {('FWE'[b]): show(dp[b]) for b in range(3)})
    print('tổng hiệu > 0:', sum(sum(dp[b][off + 1:]) for b in range(3)), '| vét cạn:', brute(s))

trace("FFF")

vòng 0: {'F': {0: 1}, 'W': {1: 1}, 'E': {-1: 1}}
vòng 1: {'F': {-1: 1, 1: 1}, 'W': {0: 1, 1: 1}, 'E': {-1: 1, 0: 1}}
vòng 2: {'F': {-1: 1, 0: 2, 1: 1}, 'W': {0: 2, 1: 1, 2: 1}, 'E': {-2: 1, -1: 1, 0: 2}}
tổng hiệu > 0: 3 | vét cạn: 3


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def countWinningSequences(self, s: str) -> int:
        MOD = 10 ** 9 + 7
        idx = {'F': 0, 'W': 1, 'E': 2}
        n = len(s)
        off = n
        W = 2 * n + 1
        dp = [[0] * W for _ in range(3)]
        a = idx[s[0]]
        for b in range(3):
            g = 0 if b == a else (1 if b == (a + 1) % 3 else -1)
            dp[b][off + g] = 1
        for ch in s[1:]:
            a = idx[ch]
            nd = [None] * 3
            for b in range(3):
                p1, p2 = dp[(b + 1) % 3], dp[(b + 2) % 3]
                summed = [(u + v) % MOD for u, v in zip(p1, p2)]
                if b == a:
                    nd[b] = summed
                elif b == (a + 1) % 3:                 # Bob thắng: hiệu +1
                    nd[b] = [0] + summed[:-1]
                else:                                  # Bob thua: hiệu −1
                    nd[b] = summed[1:] + [0]
            dp = nd
        return sum(sum(row[off + 1:]) for row in dp) % MOD

for s, exp in examples:
    got = Solution().countWinningSequences(s)
    print(got, 'OK' if got == exp else 'SAI')

3 OK
18 OK


Dịch mảng không làm mất thông tin: hiệu không thể vượt `±n`, nên ô bị đẩy ra ngoài luôn bằng 0.

## 10. Vì sao thuật toán đúng

1. **Trạng thái đủ:** điều kiện thắng chỉ phụ thuộc tổng `Σ gain`; ràng buộc không lặp chỉ phụ thuộc nước vừa đi ⇒ `(nước cuối, hiệu)` đủ để quyết định mọi khả năng tương lai.
2. **Chuyển trạng thái:** mỗi chuỗi độ dài `i + 1` hợp lệ kết thúc bằng `b` là một chuỗi độ dài `i` hợp lệ kết thúc bằng `p ≠ b` nối `b` — song ánh, hiệu tăng đúng `gain(b, a)`.
3. **Cắt tỉa (cách 4):** với `r` vòng còn lại, hiệu thay đổi tối đa `r` ⇒ `d > r` chắc thắng, `d ≤ −r` chắc không thắng; số cách hoàn thành một tiền tố bất kỳ là `2^r` (mỗi vòng chỉ cấm đúng một nước).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(3^n · n)` | `O(n)` |
| 2. Đệ quy có nhớ | `O(n² · 3 · 2)` | `O(n²)` |
| 3. DP mảng dịch | `O(n²)` (vector) | `O(n)` |
| 4. DP cắt tỉa | `O(n²)` (thường nhỏ hơn) | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'memo': memo, 'dp_shift': dp_shift, 'dp_pruned': dp_pruned,
         'submit': lambda s: Solution().countWinningSequences(s)}
edge = [("F", 1), ("W", 1), ("FF", 2), ("FW", 2), ("EEEE", 8)]
for s, exp in examples + edge:
    assert brute(s) == exp, (s, brute(s))
    for name, f in impls.items():
        assert f(s) == exp, (name, s, f(s))
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [10]:
import random
rng = random.Random(3320)
cnt = 0
for _ in range(300):
    s = ''.join(rng.choice('FWE') for _ in range(rng.randint(1, 9)))
    exp = brute(s)
    for name, f in impls.items():
        assert f(s) == exp, (name, s)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'chuỗi.')

Khớp vét cạn trên 300 chuỗi.


### 12.3. Ca lớn `n = 1000`

In [11]:
import time
big = {
    'ngẫu nhiên': ''.join(rng.choice('FWE') for _ in range(1000)),
    'toàn F': 'F' * 1000,
    'FWE lặp': 'FWE' * 333 + 'F',
}
for name, s in big.items():
    t = time.time(); a = Solution().countWinningSequences(s); t1 = time.time() - t
    t = time.time(); b = dp_pruned(s); t2 = time.time() - t
    assert a == b
    print(f"{name:12s}: {a:10d} | nộp {t1:.2f}s | cắt tỉa {t2:.2f}s")

ngẫu nhiên  :  503789141 | nộp 0.33s | cắt tỉa 0.43s


toàn F      :  857344381 | nộp 0.31s | cắt tỉa 0.29s


FWE lặp     :  833615050 | nộp 0.35s | cắt tỉa 0.50s


Kiểm tra độc lập bằng tính đối xứng: tổng số chuỗi hợp lệ của Bob là `3 · 2^{n−1}` = (số thắng) + (số hòa) + (số thua). Ta tính số hòa và số thua bằng cùng DP và kiểm tra tổng.

In [12]:
def outcome_counts(s):
    n = len(s); off = n; W = 2 * n + 1
    dp = [[0] * W for _ in range(3)]
    for b in range(3):
        dp[b][off + gain(b, IDX[s[0]])] += 1
    for ch in s[1:]:
        a = IDX[ch]
        nd = []
        for b in range(3):
            summed = [(u + v) % MOD for u, v in zip(dp[(b + 1) % 3], dp[(b + 2) % 3])]
            g = gain(b, a)
            nd.append(summed if g == 0 else ([0] + summed[:-1] if g == 1 else summed[1:] + [0]))
        dp = nd
    win = sum(sum(r[off + 1:]) for r in dp) % MOD
    tie = sum(r[off] for r in dp) % MOD
    lose = sum(sum(r[:off]) for r in dp) % MOD
    return win, tie, lose

for s in big.values():
    w, t_, l = outcome_counts(s)
    assert (w + t_ + l) % MOD == 3 * pow(2, len(s) - 1, MOD) % MOD
print('thắng + hòa + thua = 3·2^(n-1): đúng cho cả ba ca lớn.')

thắng + hòa + thua = 3·2^(n-1): đúng cho cả ba ca lớn.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên luật "Bob không lặp nước đi liền nhau"

In [13]:
def wrong_allow_repeat(s):
    total = 0
    for seq in product(range(3), repeat=len(s)):
        if sum(gain(b, IDX[a]) for b, a in zip(seq, s)) > 0:      # SAI: không kiểm tra lặp
            total += 1
    return total

print('sai :', wrong_allow_repeat("FFF"), '| đúng:', Solution().countWinningSequences("FFF"))

sai : 10 | đúng: 3


**Vì sao sai:** đề loại các chuỗi như `"WWE"`, `"EWW"`, `"WWW"`.

### Lỗi 2 — Tính cả hòa là thắng (`diff ≥ 0`)

In [14]:
def wrong_tie_counts(s):
    n = len(s); off = n
    w, t_, l = outcome_counts(s)
    return (w + t_) % MOD                                          # SAI

print('sai :', wrong_tie_counts("FFF"), '| đúng:', Solution().countWinningSequences("FFF"))

sai : 9 | đúng: 3


**Vì sao sai:** "Bob thắng" cần điểm **lớn hơn hẳn**; với `"FFF"`, chuỗi `"WFE"` (`+1, 0, −1`) hay `"FWE"`… cho hiệu 0 — là hòa, không được đếm.

### Lỗi 3 — Đảo chiều luật thắng (nghĩ `F` thắng `W`)

In [15]:
def wrong_cycle(s):
    g2 = lambda b, a: 0 if b == a else (1 if a == (b + 1) % 3 else -1)     # SAI: chiều ngược
    total = 0
    for seq in product(range(3), repeat=len(s)):
        if all(seq[i] != seq[i + 1] for i in range(len(s) - 1)) and sum(g2(b, IDX[a]) for b, a in zip(seq, s)) > 0:
            total += 1
    return total

print('sai :', wrong_cycle("FFW"), '| đúng:', Solution().countWinningSequences("FFW"))
print('(với "FWEFW" hai cách lại trùng 18 — do tính đối xứng của chuỗi, nên đừng chỉ thử ví dụ đề!)', wrong_cycle("FWEFW"))

sai : 5 | đúng: 4
(với "FWEFW" hai cách lại trùng 18 — do tính đối xứng của chuỗi, nên đừng chỉ thử ví dụ đề!) 18


**Vì sao sai:** đề: `W` (nước) thắng `F` (lửa), `F` thắng `E` (đất), `E` thắng `W`. Đảo chiều là đếm số chuỗi **thua** của Bob.

### Lỗi 4 — Chỉ so điểm của Bob với 0 (quên điểm của Alice)

In [16]:
def wrong_only_bob(s):
    total = 0
    for seq in product(range(3), repeat=len(s)):
        if all(seq[i] != seq[i + 1] for i in range(len(s) - 1)) and \
           sum(1 for b, a in zip(seq, s) if gain(b, IDX[a]) == 1) > 0:              # SAI
            total += 1
    return total

print('sai :', wrong_only_bob("FFF"), '| đúng:', Solution().countWinningSequences("FFF"))

sai : 10 | đúng: 3


**Vì sao sai:** Bob cần **nhiều điểm hơn Alice**, không chỉ "có điểm". `"WEF"`: Bob thắng vòng 1, thua vòng 2 ⇒ hòa 1–1.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm chuỗi lựa chọn sao cho tổng điểm A > B" ⇒ DP trên **hiệu** với offset.
- Ràng buộc giữa hai bước liên tiếp ⇒ thêm "lựa chọn cuối" vào trạng thái.
- Hiệu bị chặn bởi số bước còn lại ⇒ cắt tỉa chắc thắng/chắc thua.

## 15. Biến thể của bài

1. Bob không được lặp nước đi **3 lần** liên tiếp ⇒ thêm "độ dài chuỗi lặp hiện tại" vào trạng thái.
2. Đếm số chuỗi **hòa** / **thua** — cùng DP (đã dùng ở 12.3).
3. Mỗi vòng thắng được trọng số `w_i` ⇒ hiệu dịch `±w_i`, mảng rộng `Σ w_i`.
4. Hỏi xác suất thắng nếu Bob chọn ngẫu nhiên đều trong các nước hợp lệ ⇒ đáp án `/(3·2^{n−1})`.

In [17]:
# Biến thể 1: không lặp quá 2 lần liên tiếp
def no_triple(s):
    n = len(s)
    A = [IDX[c] for c in s]
    cur = {}
    for b in range(3):
        cur[(b, 1, gain(b, A[0]))] = 1
    for i in range(1, n):
        nxt = {}
        for (last, run, d), v in cur.items():
            for b in range(3):
                nr = run + 1 if b == last else 1
                if nr > 2:
                    continue
                key = (b, nr, d + gain(b, A[i]))
                nxt[key] = (nxt.get(key, 0) + v) % MOD
        cur = nxt
    return sum(v for (b, r, d), v in cur.items() if d > 0) % MOD

def no_triple_brute(s):
    return sum(1 for seq in product(range(3), repeat=len(s))
               if all(not (seq[i] == seq[i + 1] == seq[i + 2]) for i in range(len(s) - 2))
               and sum(gain(b, IDX[a]) for b, a in zip(seq, s)) > 0)

rng = random.Random(15)
for _ in range(150):
    s = ''.join(rng.choice('FWE') for _ in range(rng.randint(1, 8)))
    assert no_triple(s) == no_triple_brute(s)
print('Biến thể "không lặp quá 2 lần" khớp vét cạn.')

Biến thể "không lặp quá 2 lần" khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi chuỗi | `O(3^n n)` | Oracle |
| 2 | Đệ quy có nhớ | `O(n²)` | Dễ viết, đệ quy sâu |
| 3 | Mảng dịch | `O(n²)` | **Nộp bài** |
| 4 | Cắt tỉa | `≤ O(n²)` | Ít trạng thái hơn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "W"`.

**Bài 2 (tính tay):** `s = "FE"`.

**Bài 3 (điền TODO):** trả về **số chuỗi hòa**.

**Bài 4 (tự viết lại):** biến thể 3 — vòng `i` có trọng số `w_i`.

**Bài 5 (mini challenge):** dựng **chuỗi thắng nhỏ nhất theo thứ tự từ điển** (theo bảng chữ `E < F < W`), hoặc `""` nếu không có.

In [18]:
def ties_todo(s):
    # TODO
    raise NotImplementedError

def weighted_todo(s, w):
    # TODO
    raise NotImplementedError

def lexicographically_smallest_win_todo(s):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Bob cần thắng vòng duy nhất. (2) `E` thắng `W`. (3) `1`.

**Bài 2.** (1) Liệt kê 6 chuỗi không lặp: `FW, FE, WF, WE, EF, EW`. (2) Hiệu: `−1, 0, +2, +1, 0, −2`. (3) Thắng: `WF`, `WE` ⇒ `2`.

**Bài 3.** (1) Lấy `dp[·][off]`. (2) Hàm `outcome_counts`. (3) Vét cạn.

**Bài 4.** (1) Offset `Σ w`. (2) Dịch `±w_i`. (3) Dùng từ điển để gọn.

**Bài 5.** (1) Tính `can[i][last][d]` = có thể thắng từ trạng thái này không (DP ngược). (2) Dựng tham lam từ trái: thử nước nhỏ nhất hợp lệ còn thắng được. (3) Vét cạn đối chiếu.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().countWinningSequences("W") == 1
assert Solution().countWinningSequences("FE") == 2 == brute("FE")

def ties(s):
    return outcome_counts(s)[1]

rng = random.Random(3)
for _ in range(150):
    s = ''.join(rng.choice('FWE') for _ in range(rng.randint(1, 8)))
    exp = sum(1 for seq in product(range(3), repeat=len(s))
              if all(seq[i] != seq[i + 1] for i in range(len(s) - 1)) and sum(gain(b, IDX[a]) for b, a in zip(seq, s)) == 0)
    assert ties(s) == exp
print('Bài 1: 1 | Bài 2: 2 | Bài 3 khớp vét cạn.')

Bài 1: 1 | Bài 2: 2 | Bài 3 khớp vét cạn.


In [20]:
# Bài 4, 5
def weighted(s, w):
    cur = {}
    for b in range(3):
        cur[(b, gain(b, IDX[s[0]]) * w[0])] = 1
    for i in range(1, len(s)):
        nxt = {}
        for (last, d), v in cur.items():
            for b in range(3):
                if b != last:
                    key = (b, d + gain(b, IDX[s[i]]) * w[i])
                    nxt[key] = (nxt.get(key, 0) + v) % MOD
        cur = nxt
    return sum(v for (b, d), v in cur.items() if d > 0) % MOD

def lexicographically_smallest_win(s):
    n = len(s); A = [IDX[c] for c in s]
    order = [2, 0, 1]                      # E < F < W theo chữ cái
    from functools import lru_cache
    @lru_cache(maxsize=None)
    def can(i, last, d):                   # còn thắng được từ đây không
        if i == n:
            return d > 0
        return any(can(i + 1, b, d + gain(b, A[i])) for b in range(3) if b != last)
    if not can(0, -1, 0):
        return ""
    out, last, d = [], -1, 0
    for i in range(n):
        for b in order:
            if b != last and can(i + 1, b, d + gain(b, A[i])):
                out.append('FWE'[b]); last = b; d += gain(b, A[i])
                break
    return ''.join(out)

rng = random.Random(4)
for _ in range(150):
    s = ''.join(rng.choice('FWE') for _ in range(rng.randint(1, 7)))
    w = [rng.randint(1, 3) for _ in s]
    exp_w = sum(1 for seq in product(range(3), repeat=len(s))
                if all(seq[i] != seq[i + 1] for i in range(len(s) - 1))
                and sum(gain(b, IDX[a]) * wi for b, a, wi in zip(seq, s, w)) > 0)
    assert weighted(s, w) == exp_w
    wins = sorted(''.join('FWE'[b] for b in seq) for seq in product(range(3), repeat=len(s))
                  if all(seq[i] != seq[i + 1] for i in range(len(s) - 1)) and sum(gain(b, IDX[a]) for b, a in zip(seq, s)) > 0)
    assert lexicographically_smallest_win(s) == (wins[0] if wins else "")
print('Bài 4, 5 khớp vét cạn. Ví dụ chuỗi thắng nhỏ nhất của "FFF":', lexicographically_smallest_win("FFF"))

Bài 4, 5 khớp vét cạn. Ví dụ chuỗi thắng nhỏ nhất của "FFF": FWF


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
gain(b, a) = 0 nếu b == a; +1 nếu b == (a+1) % 3; -1 ngược lại     # F=0, W=1, E=2
dp[b] = mảng hiệu (offset n)
mỗi vòng: summed = dp[b+1] + dp[b+2]; dịch theo gain(b, a)
ans = Σ_b Σ_{d > 0} dp[b][d]
```

**Checklist:**

- [ ] Luật thắng đúng chiều (`W > F > E > W`)?
- [ ] Bob không lặp nước liền nhau (chuyển từ 2 nước khác)?
- [ ] Thắng là hiệu **> 0** (hòa không tính)?
- [ ] Offset đủ rộng `[−n, n]`?
- [ ] Modulo khi cộng?